In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime
from functools import partial

# local imports
import sys

print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)


from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.Constants import CTE as CTE

from makedf.mcstat import get_MCstat_unc

from analysis_village.cc1pi.var_configs import *

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
save_result = True
save_fig = save_result

save_fig_base_dir = "/exp/sbnd/data/users/lpelegri/Graphs/"
save_fig_dir = path.join(save_fig_base_dir, "unfolding_fake_data_tests")

if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)

# Load MC dataframe

In [ ]:
#Load CV dataframe
pot_weight_col = ('slc', 'wgt', '', '', '', '')

keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
bnb_path = "/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p_extended_syst_pruned.df"
mc_bnb_df = load_df(bnb_path, keys2load, 100, reprocess_df = False, reprocess_truth = False)
mc_evt_df = mc_bnb_df['cc1pi']
mc_nu_df = mc_bnb_df['nudf']
mc_hdr_df = mc_bnb_df['hdr']

# BNB data
print("data_tot_pot: %.3e" %(data_tot_pot))
print("data tot gates : %.3e" %(data_gates))

#Add weight column
mc_tot_pot = mc_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_evt_df))
print(len(mc_evt_df))

mc_evt_df = mc_evt_df[build_event_cumulative_masks(mc_evt_df, sideband = "")["energy"]]
mc_evt_df = (
        mc_evt_df
        .groupby(['__ntuple', 'entry', 'rec.slc..index'])
        .first()
    )
mc_evt_df = mc_evt_df.sort_index()

'''
mc_evt_df = perform_truth_matching_low_memmory(mc_evt_df, mc_nu_df)
'''
print(mc_nu_df.columns)
new_columns = []
for c in mc_nu_df.columns:
    new_columns.append(('truth',) + c + ('',) + ('',))  # prepend 'truth'
mc_nu_df.columns = pd.MultiIndex.from_tuples(new_columns)
mc_nu_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_nu_df))



In [ ]:

cols_to_keep = [
    ('nu_categ', '', ''),
    ('genie_categ', '', ''),
    ('genie_mode', '', ''),
    ('nu_categ_proton_reduced', '', ''),
    ('true_var', 'true_cos_theta_mu', ''),
    ('true_var', 'true_cos_theta_pi', ''),
    ('true_var', 'true_mu_pi_angle', ''),
    ('true_var', 'true_p_mu', ''),
    ('true_var', 'true_p_pi', ''),
    ('true_var', 'num_protons', ''),
    ('true_var', 'delta_pT', ''),
    ('true_var', 'delta_alpha_T', ''),
    ('true_var', 'delta_phi_T', ''),
    ('mu', 'dir', 'z'),
    ('cpi', 'genp', 'x'),
    ('cpi', 'genp', 'y'),
    ('cpi', 'genp', 'z'),
    ('Q2', '', ''),
    ('position', 'x', ''),
    ('position', 'y', ''),
    ('position', 'z', ''),
    ('pdg', '', ''),
    ('nmu_P_100MeV_3000MeV', '', ''),
    ('npi_P_130MeV_2000MeV', '', ''),
    ('npi_P_85MeV_10000MeV', '', ''),
    ('nprim', '', ''),
    ('nmu', '', ''),
    ('npi', '', ''),
    ('np', '', ''),
    ('nn', '', ''),
    ('iscc', '', ''),
    ('np_P_325MeV_10000MeV', '', ''),
    ('mu', 'totp', ''),
    ('mu', 'genp', 'x'),
    ('mu', 'genp', 'y'),
    ('mu', 'genp', 'z'),
    ('genweight', '', ''),
]

#Load GiBUU dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
mc_GiBUU_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_GIBUU_gen1.df", keys2load, 100, filter_df = False)
mc_GiBUU_evt_df = mc_GiBUU_df['cc1pi']
mc_GiBUU_nu_df = mc_GiBUU_df['nudf']
mc_GiBUU_hdr_df = mc_GiBUU_df['hdr']

mc_GiBUU_nu_df = mc_GiBUU_nu_df[cols_to_keep]
mc_GiBUU_evt_df = mc_GiBUU_evt_df[build_event_cumulative_masks(mc_GiBUU_evt_df, sideband = "")["energy"]]
mc_GiBUU_evt_df = (
        mc_GiBUU_evt_df
        .groupby(['__ntuple', 'entry', 'rec.slc..index'])
        .first()
    )
mc_GiBUU_evt_df.sort_index()





#genweight
#Do truth matchign
mc_GiBUU_evt_df = perform_truth_matching(mc_GiBUU_evt_df, mc_GiBUU_nu_df)

#Add weight column
mc_GiBUU_tot_pot = mc_GiBUU_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_GiBUU_tot_pot))
mc_pot_scale = data_tot_pot / mc_GiBUU_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_GiBUU_evt_df[pot_weight_col] = mc_pot_scale * mc_GiBUU_evt_df.truth.genweight.fillna(1)
mc_GiBUU_nu_df[pot_weight_col] = mc_pot_scale * mc_GiBUU_nu_df.truth.genweight

# Perform selection

In [ ]:

from analysis_village.cc1pi.HelperFunctions import HelperFunctions
HelperFunctions.print_purity(mc_evt_df, ('truth','nu_categ','','','',''))
HelperFunctions.print_purity(mc_GiBUU_evt_df, ('truth','nu_categ','','','',''))

# Plot Syts

In [ ]:
var_configs = [
    VariableConfig.all_evts(), 
    VariableConfig.muon_momentum(),
    VariableConfig.muon_direction(),
    VariableConfig.pion_momentum(),
    VariableConfig.pion_direction(),
    VariableConfig.angle_between_candidates(),
    VariableConfig.num_protons(),
    #VariableConfig.num_protons_two(),
    VariableConfig.delta_pt(),
    VariableConfig.delta_alpha_T(),
    VariableConfig.delta_phi_T()
]

In [ ]:
# choose variable
var_config = VariableConfig.angle_between_candidates()

In [ ]:

file_dir = "/exp/sbnd/data/users/lpelegri/syst/frac_cov_matrices"
flux_syst = np.load(file_dir + "/extended_flux_syst_dict_rate_ar23p.npz")
g4_syst = np.load(file_dir + "/extended_g4_syst_dict_rate_ar23p.npz")
genie_syst = np.load(file_dir + "/extended_genie_syst_dict_xsec_ar23p.npz")
mcstat_syst = np.load(file_dir + "/mcstat_syst_dict_ar23p.npz")
detvar_syst = np.load(file_dir + "/detvar_syst_dict.npz") 
cosmics_syst = np.load(file_dir + "/cosmics_syst_dict.npz")
# flat uncertainties
pot_frac_unc = 0.02
ntargets_frac_unc = 0.01

# Unfolding

In [ ]:
# --- config for Wiener-SVD unfolding ---
C_type = 2
Norm_type = 0
# DO NOT USE 0.5
#Norm_type = 0.5

# Closure test

In [ ]:
import numpy as np
from scipy.stats import chi2 as chi2_dist

def get_chi2(data, mc, cov, n_params=0):
    data = np.atleast_1d(data)
    mc   = np.atleast_1d(mc)

    if cov.shape != (len(data), len(data)):
        raise ValueError(f"Covariance shape {cov.shape} doesn't match data length {len(data)}")

    mask = (mc > 0) & np.isfinite(data) & np.isfinite(mc)

    data_f = data[mask]
    mc_f   = mc[mask]
    cov_f  = cov[np.ix_(mask, mask)]

    n_bins = mask.sum()
    ndof   = n_bins - n_params

    if ndof <= 0:
        return np.nan, ndof, np.nan

    if np.linalg.cond(cov_f) > 1e12:
        return np.nan, ndof, np.nan

    delta = mc_f - data_f
    try:
        chi2_val = delta @ np.linalg.solve(cov_f, delta)
    except np.linalg.LinAlgError:
        return np.nan, ndof, np.nan

    reduced_chi2 = chi2_val / ndof
    pval         = chi2_dist.sf(chi2_val, ndof)

    return chi2_val, ndof, pval

In [ ]:
def plot_unfolded_result(unfold, 
                         measured, 
                         models,
                         var_config, 
                         chi2_list=[],
                         textloc=[0.05, 0.55],
                         approval="internal",
                         plot_labels=["", "", ""],
                         plot=True,
                         save_fig=False, 
                         save_name=None,
                         data=False,
                         closure_test=False,
                         plot_xsec = True):

    bins = var_config.bins
    bin_centers = var_config.bin_centers
    bin_widths = np.diff(bins)
    
    if plot_xsec: 
        measured = measured*XSEC_UNIT
        for midx, mkey in enumerate(models.keys()):
            models[mkey] = XSEC_UNIT*models[mkey]
        
    # unfolded result
    Unfolded = unfold['unfold']
    UnfoldedCov = unfold["UnfoldCov"]
    if plot_xsec:
        Unfolded = Unfolded*XSEC_UNIT
        UnfoldedCov = UnfoldedCov*XSEC_UNIT*XSEC_UNIT
    
    Unfolded_perwidth = Unfolded / bin_widths

    # --- stat uncertainties
    UnfoldCov_stat = unfold['StatUnfoldCov']
    if plot_xsec:
        UnfoldCov_stat = UnfoldCov_stat*XSEC_UNIT*XSEC_UNIT

    # --- syst uncertainties
    UnfoldCov_syst = unfold['SystUnfoldCov']
    UnfoldCov_syst_frac = fraccov_from_cov(UnfoldCov_syst, Unfolded)
    if plot_xsec:
        UnfoldCov_syst = UnfoldCov_syst*XSEC_UNIT*XSEC_UNIT
    Unfold_uncert_syst = np.diag(UnfoldCov_syst)
    
    # --- decompose into norm and shape components
    # the first item in models dict is the nominal input model
    norm_model = list(models.keys())[0]
    SystUnfoldCov_norm, SystUnfoldCov_shape = Matrix_Decomp(models[norm_model], UnfoldCov_syst)
    Unfold_uncert_norm = np.sqrt(np.abs(np.diag(SystUnfoldCov_norm)))
    Unfold_uncert_shape = np.sqrt(np.abs(np.diag(SystUnfoldCov_shape)))

    # --- plot
    fig, ax = plt.subplots(figsize=(8.5, 7))
    # set err to 0 for closure test
    if closure_test:
        dummy_err = np.zeros_like(Unfolded_perwidth)
        bar_handle = plt.errorbar(bin_centers, Unfolded_perwidth, yerr=dummy_err, fmt='o', color='black')

    else:
        # plot shape uncertainty as error bars
        Unfold_uncert_shape_perwidth = Unfold_uncert_shape / bin_widths
        # Unfold_uncert_stat_shape_perwidth = Unfold_uncert_stat_perwidth + Unfold_uncert_shape_perwidth
        Unfold_uncert_stat_shape_perwidth = Unfold_uncert_shape_perwidth
        bar_handle = plt.errorbar(bin_centers, Unfolded_perwidth, yerr=Unfold_uncert_stat_shape_perwidth, fmt='o', color='black', capsize=3)

        # plot syst norm component as histogram at the bottom
        Unfold_uncert_norm_perwidth = Unfold_uncert_norm / bin_widths
        norm_handle = plt.bar(bin_centers, Unfold_uncert_norm_perwidth, width=bin_widths, label='Syst. error (norm)', alpha=0.5, color='gray')

    if data: # get stat uncertainty for data
        #Add statistical to chi2
        UnfoldCov_syst = UnfoldCov_syst + UnfoldCov_stat

        Unfold_uncert_stat = np.sqrt(np.abs(np.diag(UnfoldCov_stat)))
        Unfold_uncert_stat_per_width = Unfold_uncert_stat/bin_widths
        
        tot_err = np.sqrt(Unfold_uncert_stat_per_width**2 + Unfold_uncert_stat_shape_perwidth**2)
        Data_handle = plt.errorbar(bin_centers, Unfolded_perwidth, yerr=tot_err, fmt='o', color='black', capsize=3)
        
        handles = [bar_handle, Data_handle]
        labels = ['SBND Development Data', 'Measured Signal']
        
        
        '''
        Data_frac_unc = (1/np.sqrt(measured / XSEC_UNIT))
        # Data_stat = Unfolded_perwidth * Data_frac_unc
        Data_stat = Unfolded * Data_frac_unc
        factor = bin_widths.sum()/len(bin_widths)
        Data_stat = Data_stat #/ factor
        Data_stat_frac_unc_smeared = (unfold['AddSmear'] @ Data_frac_unc)
        # Data_frac_unc_smeared = (1/np.sqrt(unfold['AddSmear'] @ measured / xsec_unit))
        Data_stat_smeared = Unfolded_perwidth * Data_stat_frac_unc_smeared

        Data_stat_frac_cov = np.diag(Data_frac_unc**2)
        Data_stat_cov = cov_from_fraccov(Data_stat_frac_cov, Unfolded_perwidth)
        Data_stat_frac_cov_smeared = np.diag((unfold['AddSmear'] @ Data_frac_unc)**2)
        Data_stat_cov_smeared = cov_from_fraccov(Data_stat_frac_cov_smeared, Unfolded_perwidth)
        #UnfoldCov_syst = cov_from_fraccov(UnfoldCov_syst_frac, Unfolded_perwidth)
        UnfoldCov_syst = UnfoldCov_syst + Data_stat_cov
        UnfoldCov_syst_smeared = UnfoldCov_syst + Data_stat_cov_smeared
        '''
        
    # divide measured & model by bin width
    measured_perwidth = measured / bin_widths
    if data == False:
        reco_handle, = plt.step(bins, np.append(measured_perwidth, measured_perwidth[-1]), where='post', label='Measured Signal (Input)')
        
      # --- get chi2 values for each model to compare
    if len(chi2_list) == 0:
        chi2_vals = []
        p_values = []
    else:
        chi2_vals = chi2_list
    model_handles = []
    model_labels = []
    for midx, mkey in enumerate(models.keys()):
        model_smeared = unfold['AddSmear'] @ models[mkey]
        model_smeared_perwidth = model_smeared / bin_widths

        if len(chi2_list) == 0:
            # remove bins with <= 0 events
            # Fix chi2 mask logic: mask just once, store, reuse, improve clarity
            chi2_val,ndof, p_val = get_chi2(Unfolded, model_smeared, UnfoldCov_syst)
            #chi2_val, p_val = get_chi2(Unfolded_perwidth, model_smeared_perwidth, UnfoldCov_syst)
            # chi2_val, p_val = get_chi2(Unfolded_perwidth, model_smeared_perwidth, UnfoldCov_syst_smeared)
            chi2_vals.append(chi2_val)
            p_values.append(p_val)

        print("Unfolded perwidth: ", Unfolded_perwidth)
        print("Model smeared perwidth: ", model_smeared_perwidth)

        model_handle, = plt.step(bins, np.append(model_smeared_perwidth, model_smeared_perwidth[-1]), where='post')
        model_handles.append(model_handle)
        model_labels.append(f'$A_c \\otimes$ {mkey} ($\chi^2$ = {chi2_vals[midx]:.2f}/{len(bins)-1}), p-value = {p_values[midx]:.3f}')
        #model_labels.append(f'$A_c \\otimes$ {mkey} ($\chi^2$ = {chi2_vals[midx]:.2f}/{len(bins)-1})')

    # legend
    if closure_test:
        handles = [bar_handle, reco_handle] + model_handles
        labels = ['Unfolded Asimov Data', 'Measured Signal'] + model_labels
    elif data:
        handles = [bar_handle, norm_handle] + model_handles
        labels = ['Data (Shape Syst. Unc. + Stat. Unc.)', 'Norm. Syst. Unc.'] + model_labels
    else:
        handles = [bar_handle, norm_handle, reco_handle] + model_handles
        labels = ['Unfolded result', 'Norm. Syst. Unc.', 'Measured Signal'] + model_labels
    plt.legend(handles, labels, 
               loc='upper left', fontsize=12, frameon=False, ncol=1, bbox_to_anchor=(0.02, 0.98))

    plt.xlabel(var_config.var_labels[0], fontsize=20)
    plt.ylabel(var_config.xsec_label, fontsize=20)
    plt.title(plot_labels[2])
    plt.xlim(bins[0], bins[-1])
    plt.ylim(0., np.max(Unfolded_perwidth)*1.7)

    # ==== plot additions
    textloc_x, textloc_ha = get_textloc_x(Unfolded_perwidth, var_config.bins, textloc)
    textloc_y = textloc[1]
    add_approval_text(approval, textloc_x, textloc_y, textloc_ha)

    add_genie_version_text(textloc_x, textloc_y-0.1, textloc_ha)

    if var_config.var_save_name == "integrated":
        format_singlebin_plot()

    if save_fig:
        plt.savefig(save_name+fig_ext, bbox_inches='tight', dpi=dpi)

    if plot == True:
        plt.show()
    else:
        plt.close()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D 
from matplotlib.patches import Patch
from matplotlib.colors import to_rgba
import matplotlib.gridspec as gridspec
from matplotlib.legend_handler import HandlerTuple, HandlerBase

import numpy as np
from scipy.stats import chi2 as chi2_dist

def get_stat_covariance_matrix(cv_contents, sum_w2):
    """
    cv_contents: array of bin contents (sum of weights)
    sum_w2: array of the sum of the squares of the weights per bin
    """
    cv_contents = np.asarray(cv_contents)
    sum_w2 = np.asarray(sum_w2)
    n_bins = len(cv_contents)

    # 1. Variance for weighted Poisson is Sum(W^2)
    cov = np.diag(sum_w2)

    # 2. Fractional covariance: Var / (Content^2) = Sum(W^2) / (Sum W)^2
    with np.errstate(divide='ignore', invalid='ignore'):
        # This is the squared fractional error
        frac_variance = np.where(cv_contents > 0, sum_w2 / (cv_contents**2), 0.0)
        cov_frac = np.diag(frac_variance)

    # 3. Correlation matrix
    corr = np.eye(n_bins)

    return {
        "cov": cov,
        "cov_frac": cov_frac,
        "corr": corr,
    }

def plot_stacked_histogram_with_ratio(
    mc_df,
    data_df,
    config,
    cov_frac_matrix = None,
    cov_matrix = None,
    title: str = None,
    weight_column: tuple = None,
    data_pot: float = None,
    show_stats: bool = True,
    symmetric_ratio: bool = False,
    divide_by_bin_width: bool = False
):
    # 1. Pre-processing and Scaling
    slice_levels = ['__ntuple', 'entry', 'rec.slc..index']
    
    mc_data = mc_df[config.var_evt_reco_col]
    mc_types = mc_df[('truth','nu_categ','','','','')]
    mc_weights = mc_df[weight_column] if weight_column in mc_df.columns else pd.Series(1.0, index=mc_df.index)

    # --- Palette Selection ---
    present_categories = set(mc_types.dropna().unique())
    palettes = [category_colors, category_colors_pfp, proton_distinction_category_colors, genie_category_colors]
    
    chosen_map = category_colors
    max_overlap = -1
    for p in palettes:
        overlap = len(present_categories.intersection(p.keys()))
        if overlap > max_overlap:
            max_overlap = overlap
            chosen_map = p

    # 2. Sorting Categories
    category_totals = [(t, mc_weights[mc_types == t].sum()) for t in mc_types.dropna().unique()]
    signal_keys = ["CC1pi"]
    signals = sorted([x for x in category_totals if x[0] in signal_keys], key=lambda x: x[1], reverse=True)
    backgrounds = sorted([x for x in category_totals if x[0] not in signal_keys], key=lambda x: x[1], reverse=True)
    
    sorted_types = [x[0] for x in signals + backgrounds]
    stack_data_mc = [mc_data[mc_types == t].dropna() for t in sorted_types]
    stack_weights = [mc_weights[mc_types == t].loc[mc_data[mc_types == t].dropna().index] for t in sorted_types]

    # 3. Setup Figure
    fig = plt.figure(figsize=(10, 8))
    gs = gridspec.GridSpec(2, 1, height_ratios=[4, 1], hspace=0.07)
    ax_top = fig.add_subplot(gs[0])
    ax_ratio = fig.add_subplot(gs[1], sharex=ax_top)

    # 4. TOP PLOT
    max_bin_edge = config.bins[-1]
    stack_mc_clipped = [np.clip(data, config.bins[0], max_bin_edge) for data in stack_data_mc] 
        
    colors = [chosen_map.get(t, "#7f7f7f") for t in sorted_types]
    all_mc_weights = pd.concat(stack_weights)
    
    mc_sum, bins = np.histogram(
        pd.concat(stack_mc_clipped),
        bins=config.bins,
        weights=all_mc_weights
    )

    
    bin_centers = (bins[:-1] + bins[1:]) / 2
    bin_widths = np.diff(bins)
    
    # --- MC UNCERTAINTY ---
    # If a fractional covariance matrix is provided, use it
    if cov_frac_matrix is not None:
        frac_err = np.sqrt(np.diag(cov_frac_matrix))
        mc_error = frac_err * mc_sum    
    else:
        mc_sum_w2, _ = np.histogram(
            pd.concat(stack_mc_clipped),
            bins=config.bins,
            weights=all_mc_weights**2
        )
        mc_error = np.sqrt(mc_sum_w2)
        cov_matrix = get_stat_covariance_matrix(mc_sum, mc_sum_w2)["cov"]
        cov_frac_matrix = get_stat_covariance_matrix(mc_sum, mc_sum_w2)["cov_frac"]

    
 
    
    
    data = data_df[config.var_evt_reco_col].dropna()
    data_weights = data_df[weight_column] if weight_column in data_df.columns else pd.Series(1.0, index=mc_df.index)
    data_clipped = np.clip(data, config.bins[0], max_bin_edge)
    data_counts, _ = np.histogram(data_clipped, bins=bins, weights = data_weights)
    data_plot_counts = data_counts.astype(float)
    data_sum_w2, _ = np.histogram(
            data_clipped,
            bins=bins,
            weights=data_weights**2
        )
    data_errors = np.sqrt(data_sum_w2)

    # 6. STATISTICS   
    ret_stats_data_rate = get_stat_covariance_matrix(data_plot_counts , data_plot_counts)    
    chi2_val, ndof, p_val = get_chi2(data_counts, mc_sum, cov_matrix + ret_stats_data_rate["cov"])

    if divide_by_bin_width:
        mc_sum /= bin_widths
        mc_error /= bin_widths
        data_plot_counts /= bin_widths
        data_errors /= bin_widths
        # FIX FOR INDEX ERROR: Map every event to its bin width and divide weight
        new_stack_weights = []
        for i, d in enumerate(stack_mc_clipped):
            bin_indices = np.clip(np.digitize(d, bins) - 1, 0, len(bin_widths) - 1)
            new_stack_weights.append(stack_weights[i] / bin_widths[bin_indices])
        stack_weights = new_stack_weights


    ax_top.hist(stack_mc_clipped, bins=bins, stacked=True, weights=stack_weights,
                histtype='stepfilled', color=colors, alpha=0.3)
    ax_top.hist(stack_mc_clipped, bins=bins, stacked=True, weights=stack_weights,
                histtype='step', color=colors, linewidth=2)

    ax_top.bar(bin_centers, 2*mc_error, bottom=mc_sum-mc_error, width=bin_widths, 
               edgecolor='grey', facecolor='grey', alpha=0.2, linewidth=0)
    ax_top.bar(bin_centers, 2*mc_error, bottom=mc_sum-mc_error, width=bin_widths,
               edgecolor='grey', facecolor='none', hatch='////', alpha=0.5, linewidth=0)
    ax_top.errorbar(bin_centers, data_plot_counts, yerr=data_errors, xerr=bin_widths/2, 
                    fmt='ko', markersize=6, zorder=10, capsize=0)

    # 5. RATIO PLOT CALCULATION
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.divide(data_plot_counts, mc_sum, out=np.zeros_like(data_plot_counts), where=mc_sum!=0)
        ratio_error = np.divide(data_errors, mc_sum, out=np.zeros_like(data_errors), where=mc_sum!=0)
        mc_rel_error = np.divide(mc_error, mc_sum, out=np.zeros_like(mc_error), where=mc_sum!=0)
    
    # --- DYNAMIC Y-LIMIT CALCULATION ---
    # We look at the deviation from 1.0 (abs(ratio - 1)) + the error bar
    # and find the maximum such value across all bins.
    valid_ratio = (mc_sum > 0)

    # --- SYMMETRIC Y-LIMIT CALCULATION (MAXIMUM EXTENT) ---
    valid_ratio = (mc_sum > 0)
    if np.any(valid_ratio) and symmetric_ratio:
        data_extrema = np.maximum(np.abs((ratio[valid_ratio] + ratio_error[valid_ratio]) - 1), 
                                  np.abs((ratio[valid_ratio] - ratio_error[valid_ratio]) - 1))
        mc_extrema = mc_rel_error[valid_ratio]
        # Take the global maximum across all bins and all components
        max_deviation = np.max(np.maximum(data_extrema, mc_extrema))
        
        y_padding = max_deviation * 1.4
        y_padding = max(y_padding, 0.1)
        
        ax_ratio.set_ylim(1 - y_padding, 1 + y_padding)
    elif np.any(valid_ratio):
        # Distance of point (including error) from 1.0
        max_deviation = np.max(np.abs(ratio[valid_ratio] - 1) + ratio_error[valid_ratio])
        # Add 10% headroom and clip to a maximum of 0.5 (which results in ylim 0.5 to 1.5)
        y_padding = min(max_deviation * 1.4, 0.75)
        # Ensure we have a minimum padding so the plot isn't flat
        y_padding = max(y_padding, 0.1)

        ax_ratio.set_ylim(1 - y_padding, 1 + y_padding)
    else:
        ax_ratio.set_ylim(0.5, 1.5)
        
        
    ax_ratio.bar(bin_centers, 2*mc_rel_error, bottom=1-mc_rel_error, width=bin_widths,
                 edgecolor='grey', facecolor='grey', alpha=0.2, linewidth=0)
    ax_ratio.bar(bin_centers, 2*mc_rel_error, bottom=1-mc_rel_error, width=bin_widths,
                 edgecolor='grey', facecolor='none', hatch='////', alpha=0.4, linewidth=0)
    ax_ratio.errorbar(bin_centers, ratio, yerr=ratio_error, xerr=bin_widths/2, fmt='ko', markersize=6, capsize=0)
    ax_ratio.axhline(1.0, color='#d62728', linestyle='--', linewidth=2)


    # 5.5 VERTICAL CUT LINE
    cut_val = -999 # Use getattr to be safe
    draw_cut = (cut_val != -999)
    
    if draw_cut:
        # Low zorder (1) puts it behind most elements; 
        # Standard hist is usually 1, so 1.5 or 2 keeps it visible but "back"
        for ax in [ax_top, ax_ratio]:
            ax.axvline(cut_val, color='black', linestyle='--', linewidth=2, zorder=2)
        
        # Create handle for legend
        cut_hand = Line2D([0], [0], color='black', linestyle='--', linewidth=2, label='Selection Cut')
        
    
    # 7. LEGEND
    total_data_counts = len(data_clipped)
    mc_hand = [Patch(facecolor=to_rgba(chosen_map.get(t, "#7f7f7f"), 0.3), 
                     edgecolor=chosen_map.get(t, "#7f7f7f"), 
                     label=bkg_name_nice_map.get(t, t)) for t in sorted_types]

    err_label = 'MC Stat. Error'
    if show_stats:
        err_label = 'MC Total Error'
        
    err_hand = Patch(edgecolor='grey', facecolor='none', hatch='////', alpha=0.5, label = err_label)
    dat_hand = Line2D([0], [0], color='black', marker='o', linestyle='', label= 'Data', markersize=8)

    if show_stats:
        chi2_str = f"$\chi^{2}$ / ndf: {chi2_val:.2f} / {ndof} = {chi2_val/ndof:.3f}"
        p_value_str = f"$p_{{value}}$ = {p_val:.3f}"
        data_str = f'$N_{{\\mathrm{{Data}}}} = {total_data_counts}$'
        chi2_handle = Patch(color='none', label=chi2_str)
        p_value_handle = Patch(color='none', label=p_value_str)
        N_data_evts_handle = Patch(color='none', label=data_str)
    

    all_handles = mc_hand + [err_hand, dat_hand]
    all_labels = [h.get_label() for h in mc_hand] + [err_label ,'Data']
    if draw_cut:
        all_handles.append(cut_hand)
        all_labels.append(cut_hand.get_label())
    if show_stats:
        all_handles +=  [chi2_handle, p_value_handle, N_data_evts_handle]
        all_labels += [chi2_str, p_value_str, data_str]
        
    n_cols = (len(all_handles) + 3) // 4 
    
        
    leg = ax_top.legend(
        handles=all_handles, labels=all_labels,
        loc='upper center' , ncol=n_cols,
        fontsize=12, framealpha=1.0, edgecolor='black', fancybox=False, 
        borderaxespad=1, columnspacing=1.5, handlelength=1.5, handletextpad=0.5,
    )
    leg.get_frame().set_linewidth(1.5)

    plt.draw() 
    if show_stats:
        texts = leg.get_texts()
        for t in texts[-3:]:
            t.set_position((-28, 0))

    # 8. FINAL STYLING
    ax_top.set_xlim(bins[0], bins[-1])
    ax_top.set_ylim(0, ax_top.get_ylim()[1] * 1.5)

    ylabel = f'Candidate Slices'
    if divide_by_bin_width:
        ylabel += " / bin width"
    
    ax_top.set_ylabel(f"{ylabel} (POT = {data_tot_pot:.2e})")
    ax_ratio.set_ylabel("Data/MC")
    ax_ratio.set_xlabel(config.var_plot_name + " " + config.var_unit, fontsize=20)
    ax_ratio.tick_params(axis='x', which='both', direction='inout', length=6)
    ax_top.set_title("")
    
    plt.setp(ax_top.get_xticklabels(), visible=False)
    fig.align_ylabels([ax_top, ax_ratio])
    plt.subplots_adjust(top=0.92, bottom=0.12, left=0.12, right=0.95, hspace=0.07)
    
    plt.show()
    return fig, chi2_val/ndof

In [ ]:
plot_extras = True
eps = 1e-8
ratio = True
approval = "internal"
textloc = [0.05, 0.55]
ax_ylim_ratio = 1.6
breakdown_type = "topology"

unfolding_plotter = partial(
    overlay_hists,
    breakdown_type=breakdown_type,
    mc_df=mc_evt_df,
    data_df=mc_evt_df,
    intime_df=None,
    ax_ylim_ratio=ax_ylim_ratio,
    ratio=ratio,
    textloc=textloc,
    approval=approval,
    save_fig=save_fig, 
)

pot_str = get_pot_str(data_tot_pot)
plot_labels_hist = [var_config.var_labels[1], "Events / Bin (POT={})".format(pot_str), ""]

file_unfold_dir = "/exp/sbnd/data/users/lpelegri/unfold_results"
save_fig_dir = path.join(save_fig_base_dir, "unfolding_fake_data_tests/closure_test")
if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)
    

for var_config in var_configs:
                
    
    '''
    ret = unfolding_plotter(var_config=var_config,
                        plot_labels=plot_labels_hist,
                        syst=None,
                        save_name=path.join(save_fig_dir, "{}_{}.png".format(var_config.var_save_name, breakdown_type)))
    '''

    ret = signal_hists(mc_evt_df, mc_nu_df, var_config, return_data=True, plot=False)
    covariance_frac  = mcstat_syst[var_config.var_save_name]
    Covariance = cov_from_fraccov(covariance_frac, ret["nevts_sel_reco"])


    print(covariance_frac)
    print(Covariance)
    if plot_extras:
        fig, chi2 = plot_stacked_histogram_with_ratio(
                mc_evt_df,
                mc_evt_df,
                var_config,
                cov_frac_matrix = mcstat_syst[var_config.var_save_name],
                cov_matrix = Covariance,
                weight_column = ('slc', 'wgt','','','',''),
                data_pot = data_tot_pot,
                show_stats = False,
                symmetric_ratio =  True,
                divide_by_bin_width = True
                )
     
    
        save_name = path.join(save_fig_dir, "{}_closure_breakdown.pdf".format(var_config.var_save_name))
        fig.savefig(save_name, format='pdf', bbox_inches='tight')
    
    reco_vs_true, _, _ = np.histogram2d(ret["var_sel_truth"], 
                                            ret["var_sel_reco"], 
                                            weights=ret["wgt_sel_reco"], 
                                            bins=[var_config.bins, var_config.bins]) # Fix: wrap them!

    print(reco_vs_true)
    eff = ret["nevts_sel_truth"] / ret["nevts_allmc"]
    print(ret["nevts_sel_truth"])
    print(ret["nevts_allmc"])
    print(eff)
    
    response = get_response_matrix(reco_vs_true, eff)

    if plot_extras:
        '''
        save_fig_name = "{}/{}_reco_vs_true".format(save_fig_dir, var_config.var_save_name)
        plot_heatmap(reco_vs_true, 
                     var_config.bins, 
                     plot_labels=[var_config.var_labels[2], var_config.var_labels[1], "Smearing"],
                     verbose=True,
                     save_fig=False, 
                     save_name=save_fig_name)
        '''
        evts_signal_truth, _, _ = plt.hist(ret["var_allmc"], bins=var_config.bins, weights=ret["wgt_allmc"], histtype="step", label="True Signal")
        nevts_signal_sel_reco, _, _ = plt.hist(ret["var_sel_reco"], bins=var_config.bins, weights=ret["wgt_sel_reco"], histtype="step", label="Reco Selected Signal", color="k")
        nevts_signal_sel_truth, _, _ = plt.hist(ret["var_sel_truth"], bins=var_config.bins, weights=ret["wgt_sel_truth"], histtype="step", label="True Selected Signal")    

        save_fig_name = "{}/{}_response_matrix".format(save_fig_dir, var_config.var_save_name)
        
        plot_heatmap(response, 
                     var_config.bins, 
                     plot_labels=[var_config.var_labels[2], var_config.var_labels[1], "Response"],
                     save_fig=save_fig, 
                     verbose=True,
                     save_name=save_fig_name)

    measured = ret["nevts_sel_reco"]  
    model    = ret["nevts_allmc"] 

    print(response)
    print(model)
    print(measured)
    print(Covariance)
    print(C_type)
    print(Norm_type)
    unfold = WienerSVD(response, model, measured, Covariance, C_type, Norm_type)

    models = {"SBND Baseline model": model}
    save_name = "{}/{}_closure_test_output.pdf".format(save_fig_dir,var_config.var_save_name)
    print(models)
    
    plot_unfolded_result(unfold, 
                         measured, 
                         models, 
                         var_config,
                         save_fig=save_fig, 
                         save_name=save_name,
                         closure_test=True)

    save_fig_name = "{}/{}_{}_AC_matrix".format(save_fig_dir, var_config.var_save_name, "closure_test")
    plot_heatmap(unfold["AddSmear"], 
                 var_config.bins, 
                 plot_labels=[var_config.var_labels[2], var_config.var_labels[1], "$A_c$"],
                 save_fig=save_fig, 
                 save_name=save_fig_name)
    
    np.savez(file_unfold_dir + "/unfolding_result_" + var_config.var_save_name + ".npz", **unfold)

# Fake Data Test

In [ ]:
from analysis_village.cc1pi.systematics.fake_data_test_config import FakeDataWeights
fake_weight_obj = FakeDataWeights(mc_evt_df, mc_nu_df, var_config)
# test_name = "bump"

test_configs = {
    #"mec_test": [0.5, "MEC Scale"],
    "res_test_1p2": [1.2, "Res Scale"],
    "res_test_0p8": [0.8, "Res Scale"],
    "qe_test_0p5": [0.5, "QE Scale"],
    "qe_test_1p5": [1.5, "QE Scale"],
    #"1pi1p_test": [1.2, "1pi1pScale"],
    #"sig_test": [1.2, "Signal Scale"],
    #"q2_test_alpha_0.3": [0.2, "$Q^2$ tilt"],
    "costh_weight_scale_0p7": [0.7, "Forward Muon Scale"],
    "costh_weight_scale_1p3": [1.3, "Forward Muon Scale"],
    "pion_P_tilt_alpha_0p4": [0.4, "Pion Momentum Tilt"],
    "pion_P_tilt_alpha_0p8": [0.8, "Pion Momentum Tilt"],
    "pion_P_weight_scale_0p7": [0.7, "Pion Momentum Scale"],
    "pion_P_weight_scale_1p3": [1.3, "Pion Momentum Scale"],

}



In [ ]:
#from analysis_village.cc1pi.systematics.fake_data_test_config import FakeDataWeights
weight_col = ("slc","wgt","","","","")

plot_overlay = False



for var_config in var_configs:

    
    ret = signal_hists(mc_evt_df, mc_nu_df, var_config, return_data=True, plot=False)
    reco_vs_true, _, _ = np.histogram2d(ret["var_sel_truth"], 
                                            ret["var_sel_reco"], 
                                            weights=ret["wgt_sel_reco"], 
                                            bins=[var_config.bins, var_config.bins]) # Fix: wrap them!
    eff = ret["nevts_sel_truth"] / ret["nevts_allmc"]
    response = get_response_matrix(reco_vs_true, eff)
    
    covariance_frac = genie_syst[var_config.var_save_name] + mcstat_syst[var_config.var_save_name]# + GiBUU_cov_matrices[var_config.var_save_name]["cov_frac"]
    covariance = cov_from_fraccov(covariance_frac, ret["nevts_sel_reco"])

    
    plot_heatmap(covariance, var_config.bins, 
             plot_labels=[var_config.var_labels[2], var_config.var_labels[1], "Covariance"], 
             save_fig=False)

    
    print(ret["nevts_sel_reco"])
    
    syst = np.sqrt(np.diag(covariance_frac))
    
    for test_name, configs in test_configs.items():
        
        save_fig_dir = path.join(save_fig_base_dir, f"unfolding_fake_data_tests/fake_data_tests/{test_name}")
        if save_fig:
            if not path.exists(save_fig_dir):
                makedirs(save_fig_dir)
            
        weights_fake_data, weight_fakedata_signal_truth = fake_weight_obj.get_weights(test_name, scale_factor=configs[0])
        fakedata_evt_df = mc_evt_df.copy()
        fakedata_evt_df[weight_col] *= weights_fake_data
        
        evtdf_signal = mc_evt_df[mc_evt_df.truth.nu_categ == "CC1pi"]
        nudf_signal = mc_nu_df[mc_nu_df.truth.nu_categ == "CC1pi"]
        ret = signal_hists(mc_evt_df, mc_nu_df, var_config, return_data=True, plot=False)
        
        nevts_fakedata_reco, _ = np.histogram(ret["var_allsel_reco"], bins=var_config.bins, weights=weights_fake_data * ret["wgt_allsel_reco"])
        nevts_nomdata_signal_truth, _ = np.histogram(ret["var_allmc"], bins=var_config.bins, weights= ret["wgt_allmc"])
        nevts_fakedata_signal_truth, _ = np.histogram(ret["var_allmc"], bins=var_config.bins, weights=weight_fakedata_signal_truth*ret["wgt_allmc"])
        
        print(syst)

        
        ret = overlay_hists(breakdown_type=breakdown_type,
                            mc_df=mc_evt_df,
                            data_df=fakedata_evt_df,
                            intime_df=None,
                            var_config=var_config,
                            plot_labels=plot_labels_hist,
                            ax_ylim_ratio=ax_ylim_ratio,
                            ratio=ratio,
                            syst=syst,
                            textloc=textloc,
                            approval=approval,
                            plot=plot_overlay,
                            save_fig=False, 
                            save_name=save_fig_name)
        
        
        save_fig_name = f"{save_fig_dir}/{var_config.var_save_name}_breakdown.pdf"
        fig, chi2 = plot_stacked_histogram_with_ratio(
                mc_evt_df,
                fakedata_evt_df,
                var_config,
                cov_frac_matrix = covariance_frac,
                cov_matrix = covariance,
                weight_column = ('slc', 'wgt','','','',''),
                data_pot = data_tot_pot,
                show_stats = False,
                symmetric_ratio =  True,
                divide_by_bin_width = True
                )
        save_name = save_fig_name
        fig.savefig(save_name, format='pdf', bbox_inches='tight')
    
        measured = (ret["total_data"] - ret["total_mc_bkgd"]) 
        model = nevts_nomdata_signal_truth

        
        print(response)
        print(model)
        print(measured)
        print(covariance)
        print(C_type)
        print(Norm_type)
        
        unfold = WienerSVD(response, model, measured, covariance, C_type, Norm_type)
        
        models = {"SBND Baseline Model": model, 
                "Fake Data": nevts_fakedata_signal_truth}
        
        save_fig_name = f"{save_fig_dir}/{var_config.var_save_name}_unfolded_event_rates"
        plot_unfolded_result(unfold, 
                            measured, 
                            models, 
                            var_config,
                            plot_labels=["", "", configs[1]],
                            save_fig=save_fig, 
                            save_name=save_fig_name,
                            closure_test=False)
        
        save_fig_name = f"{save_fig_dir}/{var_config.var_save_name}-add_smear"
        plot_heatmap(unfold["AddSmear"], 
                    var_config.bins, 
                    plot_labels=[var_config.var_labels[2], var_config.var_labels[1], "$A_c$"],
                    save_fig=False, 
                    save_name=save_fig_name)    

# GIBUU Test

In [ ]:
def get_stat_covariance_matrix(cv_contents, sum_w2):
    """
    cv_contents: array of bin contents (sum of weights)
    sum_w2: array of the sum of the squares of the weights per bin
    """
    cv_contents = np.asarray(cv_contents)
    sum_w2 = np.asarray(sum_w2)
    n_bins = len(cv_contents)

    # 1. Variance for weighted Poisson is Sum(W^2)
    cov = np.diag(sum_w2)

    # 2. Fractional covariance: Var / (Content^2) = Sum(W^2) / (Sum W)^2
    with np.errstate(divide='ignore', invalid='ignore'):
        # This is the squared fractional error
        frac_variance = np.where(cv_contents > 0, sum_w2 / (cv_contents**2), 0.0)
        cov_frac = np.diag(frac_variance)

    # 3. Correlation matrix
    corr = np.eye(n_bins)

    return {
        "cov": cov,
        "cov_frac": cov_frac,
        "corr": corr,
    }

In [ ]:
evt_df_save = mc_evt_df
GiBUU_evt_df_save = mc_GiBUU_evt_df

In [ ]:
muon_threshold = 1 #3

def TruthInFV(data):
    xmax = 190.
    zmin = 10.
    zmax = 450.
    ymax_highz = 100.
    pass_xz = (np.abs(data.x) < xmax) & (data.z > zmin) & (data.z < zmax)
    pass_y = ((data.z < 250) & (np.abs(data.y) < 190.)) | ((data.z > 250) & (data.y > -190.) & (data.y < ymax_highz))
    return pass_xz & pass_y

def IsNu(df):
    is_numu = abs(df.pdg) == 14
    is_nue = abs(df.pdg) == 12
    return is_numu | is_nue   

def isCC1Pi(df): # definition
    is_1pi1mu = (df.nmu_P_100MeV_3000MeV == 1) & (df.npi_P_130MeV_2000MeV == 1) & (df.npi_P_85MeV_10000MeV == 1)
    is_NpiNmuNnNp = df.nprim - df.nmu - df.npi - df.np - df.nn == 0
    # Initialize full theta mask (False by default)
    is_theta = pd.Series(False, index=df.index)
    low_energy_muon = df.mu.totp < muon_threshold
    
    # Only compute angles where needed
    df_sel = df.loc[is_1pi1mu]
    
    if len(df_sel) > 0:
        cpi_vec = df_sel.loc[:, ('cpi','genp',['x','y','z'])].to_numpy()
        mu_vec  = df_sel.loc[:, ('mu','genp',['x','y','z'])].to_numpy()
     

        mu_mag  = np.linalg.norm(mu_vec, axis=1)
        cpi_mag = np.linalg.norm(cpi_vec, axis=1)
        dot     = np.sum(mu_vec * cpi_vec, axis=1)

        cos_theta = dot / np.clip(mu_mag * cpi_mag, 1e-12, None)
        theta     = np.arccos(np.clip(cos_theta, -1.0, 1.0))
        
        # Assign back using the SAME index subset
        is_theta.loc[df_sel.index] = theta < CTE.max_angle_between_candidates
    
    #return is_1pi1mu & is_NpiNmuNnNp & is_theta & is_mu_contained
    return is_1pi1mu & is_NpiNmuNnNp & is_theta & low_energy_muon

  
def add_nu_categ_column(df, is_truth_df = False):
    if(is_truth_df):
        truth_df = df.truth
    else:
        truth_df = df.truth # Make a copy to safely assign

    is_inside_fv = TruthInFV(truth_df.position)
    is_nu = IsNu(truth_df)
    is_signal = isCC1Pi(truth_df)
    is_cc = truth_df.iscc.fillna(0).astype(bool)
    is_nu_mu_cc = is_cc & (abs(truth_df.pdg.fillna(0).astype(int)) == 14)


    nu_categ = pd.Series("none", index=truth_df.index, dtype="object")
    # Apply categories
    nu_categ[~is_nu] = "cosmic"
    nu_categ[is_nu & ~is_inside_fv] = "out_AV_nu"
    nu_categ[is_nu & is_inside_fv & ~is_cc] = "NC"
    nu_categ[is_nu & is_inside_fv & is_cc & (abs(truth_df.pdg) == 12)] = "CC_e"
    nu_categ[is_nu & is_inside_fv & is_nu_mu_cc & (truth_df.npi_P_85MeV_10000MeV == 0) & (truth_df.np_P_325MeV_10000MeV == 1)] = "CC_mu_0pi_1p"
    nu_categ[is_nu & is_inside_fv & is_nu_mu_cc & (truth_df.npi_P_85MeV_10000MeV == 0) & (truth_df.np_P_325MeV_10000MeV > 1)] = "CC_mu_0pi_2p"
    nu_categ[is_nu & is_inside_fv & is_nu_mu_cc & (truth_df.npi_P_85MeV_10000MeV == 0) & (truth_df.np_P_325MeV_10000MeV == 0)] = "CC_mu_0pi_0p"
    nu_categ[is_nu & is_inside_fv & is_nu_mu_cc & (truth_df.npi_P_85MeV_10000MeV > 1)] = "CC_mu_2pi"
    nu_categ[is_nu & is_inside_fv & is_nu_mu_cc & is_signal] = "CC1pi"
    nu_categ[is_nu & is_inside_fv & is_nu_mu_cc & ~is_signal & (truth_df.npi_P_85MeV_10000MeV == 1)] = "other_CC1pi"
    
    
    if(is_truth_df):
        df['truth','nu_categ','','','',''] = nu_categ
    else:
        df[('truth', 'nu_categ', '', '','')] = nu_categ 
    return df


In [ ]:
def get_GiBUU_univ(evtdf=None, 
                    nudf=None, 
                    evtdf_GiBUU=None, 
                    nudf_GiBUU=None, 
                    var_config=None, 
                    syst_name="", 
                    cov_type = "rate",
                    bkgd_subtract=True):
    """
    for the GENIE uncertainty on the xsec measurement
    """
    bins = var_config.bins

    evtdf_signal = evtdf[evtdf.truth.nu_categ == "CC1pi"]
    evtdf_signal_GiBUU = evtdf_GiBUU[evtdf_GiBUU.truth.nu_categ == "CC1pi"]

    
    # reco variable histogram, topology breakdown
    evtdf_div_topo = [evtdf[evtdf.truth.nu_categ == mode]for mode in topology_list]
    evtdf_div_topo_GiBUU = [evtdf_GiBUU[evtdf_GiBUU.truth.nu_categ == mode]for mode in topology_list]

    if nudf is not None:
        nudf_signal = nudf[nudf.truth.nu_categ == "CC1pi"]
        nudf_GiBUU_signal = nudf_GiBUU[nudf_GiBUU.truth.nu_categ == "CC1pi"]
        
    ret = signal_hists(evtdf, nudf, var_config, return_data=True, plot=False)
    ret_GiBUU = signal_hists(evtdf_GiBUU, nudf_GiBUU, var_config, return_data=True, plot=False)
    
    univ_events = []
    # ---- uncertainty on the signal rate ----
    if cov_type == "xsec":
        #Obtain the GiBUU response matrix
        if len(bins) == 2:
            reco_vs_true = np.array([[1.0]])
        else:
            reco_vs_true, _, _ = np.histogram2d(
                ret_GiBUU["var_sel_truth"], 
                ret_GiBUU["var_sel_reco"], 
                weights=ret_GiBUU["wgt_sel_truth"], # Use clipped weights here
                bins=bins
            )

        # efficiency for GiBUU
        signal_allmc_univ, _ = np.histogram(ret_GiBUU["var_allmc"],
                                           weights=ret_GiBUU["wgt_allmc"],
                                           bins=bins)
        signal_sel_univ, _ = np.histogram(ret_GiBUU["var_sel_truth"],
                                           weights=ret_GiBUU["wgt_sel_truth"],
                                           bins=bins)
        eff = signal_sel_univ / signal_allmc_univ

        response_univ = get_response_matrix(reco_vs_true, eff)
        signal_univ = response_univ @ ret["nevts_allmc"] # note that we multiply the CV signal rate!
        # signal_univ = signal_cv

    elif cov_type == "rate":            
        signal_univ, _ = np.histogram(ret_GiBUU["var_sel_reco"], 
                                      weights=ret_GiBUU["wgt_sel_reco"],
                                          bins=bins)
    else:
        raise ValueError("Invalid covariance type: {}, choose xsec or rate".format(cov_type))
        



    #BKG SUBSTRACT
    for i in range(1, len(evtdf_div_topo_GiBUU)):
        this_evtdf = evtdf_div_topo[i]
        this_evtdf_GiBUU = evtdf_div_topo_GiBUU[i]
    
        var, wgt = get_clipped_evts(this_evtdf, var_config.var_evt_reco_col, bins)
        var_GiBUU, wgt_GiBUU = get_clipped_evts(this_evtdf_GiBUU, var_config.var_evt_reco_col, bins)
        
        background_cv, _   = np.histogram(var, bins=bins, weights=wgt)
        background_univ, _ = np.histogram(var_GiBUU, bins=bins, weights=wgt_GiBUU)

        if bkgd_subtract:
            signal_univ += (background_univ - background_cv)
        else:
            signal_univ += background_univ

    univ_events.append(signal_univ)    
    univ_events = np.array(univ_events)

    if bkgd_subtract:
        cv_events = ret["nevts_sel_reco"]
    else:
        cv_events = ret["nevts_allsel_reco"]

    return univ_events, cv_events

In [ ]:
mc_evt_df = evt_df_save[(evt_df_save.slc.measure_var.reco_p_mu > 0.1) & (evt_df_save.slc.measure_var.reco_p_mu < muon_threshold) & (evt_df_save.slc.measure_var.TLE_p_pi > 0.13) & (evt_df_save.slc.measure_var.TLE_p_pi < 2)]
mc_GiBUU_evt_df = GiBUU_evt_df_save[(GiBUU_evt_df_save.slc.measure_var.reco_p_mu > 0.1) & (GiBUU_evt_df_save.slc.measure_var.reco_p_mu < muon_threshold) & (GiBUU_evt_df_save.slc.measure_var.TLE_p_pi > 0.13) & (GiBUU_evt_df_save.slc.measure_var.TLE_p_pi < 2)]  


mc_evt_df = add_nu_categ_column(mc_evt_df, False)
mc_GiBUU_evt_df = add_nu_categ_column(mc_GiBUU_evt_df, False)

mc_nu_df = add_nu_categ_column(mc_nu_df, True)
mc_GiBUU_nu_df = add_nu_categ_column(mc_GiBUU_nu_df, True)


print(muon_threshold)  
print(mc_nu_df[mc_nu_df.truth.mu.totp > muon_threshold].truth.nu_categ)
#print(mc_GiBUU_evt_df[mc_GiBUU_evt_df.slc.measure_var.reco_p_mu > muon_threshold])

In [ ]:

GiBUU_cov_matrices = {}
for var_config in var_configs:
    univ_events, cv_events = get_GiBUU_univ(evtdf=mc_evt_df, 
                        nudf=mc_nu_df, 
                        evtdf_GiBUU=mc_GiBUU_evt_df, 
                        nudf_GiBUU=mc_GiBUU_nu_df, 
                        var_config=var_config, 
                        syst_name="GiBUU", 
                        cov_type = "xsec",
                        bkgd_subtract=True)
    ret_GiBUU = get_covariance_matrix(univ_events, cv_events)
    GiBUU_cov_matrices[var_config.var_save_name] = ret_GiBUU

for var_config in var_configs:
    fig, ax = plt.subplots(figsize=(9, 6))
    box = ax.get_position()
    ax.set_position([box.x0, box.y0, box.width * 0.8, box.height])
    
    syst = GiBUU_cov_matrices[var_config.var_save_name]["cov_frac"]
    syst_uncert = np.sqrt(np.diag(syst))
    values = syst_uncert * 1e2
    
    ax.hist(
        var_config.bin_centers,
        bins=var_config.bins,
        weights=values,
        histtype="step",
        linewidth=3,
        color="k",
        label="GiBUU"
    )
    
    ax.set_xlim(var_config.bins[0], var_config.bins[-1])
    ax.set_ylim(0, max(values) * 1.6)
    ax.set_xlabel(var_config.var_labels[1])
    ax.set_ylabel("Uncertainty [%]")
    ax.legend(fontsize=10, frameon=True, edgecolor='gray')
    ax.grid(which='major', linestyle='-', linewidth=0.7, alpha=0.7)
    ax.grid(which='minor', linestyle=':', linewidth=0.5, alpha=0.5)
    ax.minorticks_on()
    plt.show()

In [ ]:
test_name = "GiBUU"

file_dir = "/exp/sbnd/data/users/lpelegri/syst/frac_cov_matrices"
#file_dir_signal = "/exp/sbnd/data/users/lpelegri/syst/frac_cov_matricesfrac_cov_matrices_main_res_knobs"
file_dir_signal = "/exp/sbnd/data/users/lpelegri/syst/frac_cov_matrices_highP_mu_knobs"

genie_signal_syst = np.load(file_dir_signal + "/extended_genie_syst_dict_xsec_ar23p.npz")
genie_syst = np.load(file_dir + "/extended_genie_syst_dict_xsec_ar23p.npz")
mcstat_syst = np.load(file_dir + "/mcstat_syst_dict_ar23p.npz")

use_GiBUU_truth = True
test_name = "GiBUU"
save_fig_dir = path.join(save_fig_base_dir, f"unfolding_fake_data_tests/fake_data_tests/{test_name}")
if not path.exists(save_fig_dir):
    makedirs(save_fig_dir)
                
for var_config in var_configs:
    print(genie_syst[var_config.var_save_name])
    
    unfolded_plot_labels = [var_config.var_labels[0], var_config.xsec_label]
    smearmat_plot_labels = [var_config.var_labels[2], var_config.var_labels[1]]

    
    ret = signal_hists(mc_evt_df, mc_nu_df, var_config, return_data=True, plot=False)
    reco_vs_true, _, _ = np.histogram2d(ret["var_sel_truth"], 
                                                ret["var_sel_reco"], 
                                                weights=ret["wgt_sel_reco"], 
                                                bins=[var_config.bins, var_config.bins]) # Fix: wrap them!
    eff = ret["nevts_sel_truth"] / ret["nevts_allmc"]
    response = get_response_matrix(reco_vs_true, eff)
    
    
    
    covariance_frac = genie_syst[var_config.var_save_name] + mcstat_syst[var_config.var_save_name] 
    print( "Start cov matrix")
    print( np.sqrt(np.diag( genie_syst[var_config.var_save_name])))
    print( "-------------")
    print( np.sqrt(np.diag(genie_signal_syst[var_config.var_save_name])))
    print("x20:")
    print( np.sqrt(np.diag(20*genie_signal_syst[var_config.var_save_name])))
    print( "-------------")
    
    print(  np.sqrt(np.diag( genie_syst[var_config.var_save_name] + 20*genie_signal_syst[var_config.var_save_name])))
    print( "-------------")
#+ GiBUU_cov_matrices[var_config.var_save_name]["cov_frac"] #
    multiply_factor = 2
    covariance_frac = genie_syst[var_config.var_save_name] + mcstat_syst[var_config.var_save_name] #+ genie_signal_syst[var_config.var_save_name]*multiply_factor*multiply_factor -  genie_signal_syst[var_config.var_save_name]
    print( np.sqrt(np.diag(covariance_frac)))
    
    
    weight_column = ('slc', 'wgt','','','','')
    covariance = cov_from_fraccov(covariance_frac, ret["nevts_sel_reco"])
    
    plot_heatmap(covariance, var_config.bins, 
             plot_labels=[var_config.var_labels[2], var_config.var_labels[1], "Covariance"], 
             save_fig=False)
    
    

    
    data = mc_GiBUU_evt_df[var_config.var_evt_reco_col].dropna()
    data_weights = mc_GiBUU_evt_df[weight_column] 
    mc_clipped = np.clip(data, var_config.bins[0], var_config.bins[-1]-0.0000001)
    mc_sum, bins = np.histogram(
        data,
        bins=var_config.bins,
        weights=data_weights
    )
    mc_sum_w2, _ = np.histogram(
            data,
            bins=var_config.bins,
            weights=data_weights**2
    )
    GiBUU_stat_cov = get_stat_covariance_matrix(mc_sum, mc_sum_w2)["cov"]
    
    ret = signal_hists(mc_evt_df, mc_nu_df, var_config, return_data=True, plot=True)
    ret_GIBUU = signal_hists(mc_GiBUU_evt_df, mc_GiBUU_nu_df, var_config, return_data=True, plot=True)
   
    nevts_reco, _ = np.histogram(ret["var_allsel_reco"], bins=var_config.bins, weights= ret["wgt_allsel_reco"])
    nevts_fakedata_reco, _ = np.histogram(ret_GIBUU["var_allsel_reco"], bins=var_config.bins, weights= ret_GIBUU["wgt_allsel_reco"])

    nevts_signal_reco, _ = np.histogram(ret["var_sel_reco"], bins=var_config.bins, weights= ret["wgt_sel_reco"])
    nevts_signal_fakedata_reco, _ = np.histogram(ret_GIBUU["var_sel_reco"], bins=var_config.bins, weights= ret_GIBUU["wgt_sel_reco"])
    nevts_nomdata_signal_truth, _ = np.histogram(ret["var_allmc"], bins=var_config.bins, weights= ret["wgt_allmc"])
    nevts_fakedata_signal_truth, _ = np.histogram(ret_GIBUU["var_allmc"], bins=var_config.bins, weights= ret_GIBUU["wgt_allmc"])
    
    bins = var_config.bins
    bin_widths = np.diff(bins)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8), sharex=True)
    # Main plot
    ax1.hist(var_config.bin_centers, var_config.bins, weights=nevts_reco, histtype="step", label="all selected events")
    ax1.hist(var_config.bin_centers, var_config.bins, weights=nevts_fakedata_reco, histtype="step", label="alternative MC selected events")
    ax1.hist(var_config.bin_centers, var_config.bins, weights=nevts_nomdata_signal_truth, histtype="step", label="nominal MC, all signal")
    ax1.hist(var_config.bin_centers, var_config.bins, weights=nevts_fakedata_signal_truth, histtype="step", label="alternative MC, all signal")
    ax1.legend()
    ax1.set_ylabel("Events")
    
    # Ratio plot
    ratio_nom      = np.divide(nevts_signal_reco, nevts_nomdata_signal_truth,
                               out=np.zeros_like(nevts_reco), where=nevts_nomdata_signal_truth != 0)
    ratio_fakedata = np.divide(nevts_signal_fakedata_reco, nevts_fakedata_signal_truth,
                               out=np.zeros_like(nevts_signal_fakedata_reco), where=nevts_fakedata_signal_truth != 0)
    ratio_reco     = np.divide(nevts_signal_reco, nevts_signal_fakedata_reco,
                               out=np.zeros_like(nevts_reco), where=nevts_signal_fakedata_reco != 0)
    ratio_truth    = np.divide(nevts_nomdata_signal_truth, nevts_fakedata_signal_truth,
                               out=np.zeros_like(nevts_nomdata_signal_truth), where=nevts_fakedata_signal_truth != 0)
    ratio_reco_vs_truth = np.divide(ratio_reco, ratio_truth,
                                    out=np.zeros_like(ratio_reco), where=ratio_truth != 0)
    
    lw = 2.5
    ax2.hist(var_config.bin_centers, var_config.bins, weights=ratio_nom, histtype="step",
             color="royalblue", linewidth=lw, label="genie reco signal / genie truth signal")
    ax2.hist(var_config.bin_centers, var_config.bins, weights=ratio_fakedata, histtype="step",
             color="darkorange", linewidth=lw, label="GiBUU reco signal / GiBUU truth signal")
    ax2.hist(var_config.bin_centers, var_config.bins, weights=ratio_reco, histtype="step",
             color="crimson", linestyle="dashed", linewidth=lw, label="genie reco / GiBUU reco")
    ax2.hist(var_config.bin_centers, var_config.bins, weights=ratio_truth, histtype="step",
             color="forestgreen", linestyle="dashed", linewidth=lw, label="genie truth / GiBUU truth")
    ax2.hist(var_config.bin_centers, var_config.bins, weights=ratio_reco_vs_truth, histtype="step",
             color="purple", linestyle="dashed", linewidth=lw, label="ratio reco / ratio truth")
    ax2.axhline(1, color="black", linestyle="--", linewidth=0.8)
    ax2.legend()
    ax2.set_ylabel("Ratio")
    ax2.set_xlabel(var_config.var_save_name)
    
    plt.tight_layout()
    plt.show()

    syst = np.sqrt(np.diag(covariance_frac))
    ret = overlay_hists(breakdown_type=breakdown_type,
                            mc_df=mc_evt_df,
                            data_df=mc_GiBUU_evt_df,
                            intime_df=None,
                            var_config=var_config,
                            plot_labels=plot_labels_hist,
                            ax_ylim_ratio=ax_ylim_ratio,
                            ratio=ratio,
                            syst=syst,
                            textloc=textloc,
                            approval=approval,
                            save_fig=False, 
                            save_name=None)


    ret_GiBUU = overlay_hists(breakdown_type=breakdown_type,
                            mc_df=mc_GiBUU_evt_df,
                            data_df=mc_GiBUU_evt_df,
                            intime_df=None,
                            var_config=var_config,
                            plot_labels=plot_labels_hist,
                            ax_ylim_ratio=ax_ylim_ratio,
                            ratio=ratio,
                            syst=syst,
                            textloc=textloc,
                            approval=approval,
                            save_fig=False, 
                            save_name=None)
    
    measured = (ret["total_data"] - ret["total_mc_bkgd"]) 
    if use_GiBUU_truth:
        measured = (ret["total_data"] - ret_GiBUU["total_mc_bkgd"]) 
    
    model = nevts_nomdata_signal_truth
    unfold = WienerSVD(response, model, measured, covariance, C_type, Norm_type)

    unfold['StatUnfoldCov'] = unfold["CovRotation"]@ GiBUU_stat_cov @ unfold["CovRotation"].T
    
    models = {"SBND Baseline Model": model, 
              "Fake Data": nevts_fakedata_signal_truth}

    save_fig = True
    save_fig_name = f"{save_fig_dir}/{var_config.var_save_name}_unfolded_event_rates"
    if use_GiBUU_truth:
        save_fig_name = f"{save_fig_dir}/{var_config.var_save_name}_unfolded_event_rates_GiBUU_truth"
          
    print(save_fig_name)
    plot_unfolded_result(unfold, 
                         measured, 
                         models, 
                         var_config,
                         save_fig=save_fig, 
                         save_name=save_fig_name,
                         closure_test=False,
                         data = True)
    
    save_fig_name = "{}/{}-{}-add_smear".format(save_fig_dir, var_config.var_save_name, "closure_test")
    
    plot_heatmap(unfold["AddSmear"], 
                 var_config.bins, 
                 plot_labels=[var_config.var_labels[2], var_config.var_labels[1], "$A_c$"],
                 save_fig=False, 
                 save_name=save_fig_name)

# Bump test

In [ ]:
for var_config in var_configs:

    test_name = "bump_0p5_0p05_0p1"
    
    save_fig_dir = path.join(save_fig_base_dir, f"unfolding_fake_data_tests/fake_data_tests/{test_name}")
    if save_fig:
        if not path.exists(save_fig_dir):
            makedirs(save_fig_dir)
            
    fake_weight_obj = FakeDataWeights(mc_evt_df, mc_nu_df, var_config)
    unfolded_plot_labels = [var_config.var_labels[0], var_config.xsec_label]
    smearmat_plot_labels = [var_config.var_labels[2], var_config.var_labels[1]]
    
    ret = signal_hists(mc_evt_df, mc_nu_df, var_config, return_data=True, plot=False)
    reco_vs_true, _, _ = np.histogram2d(ret["var_sel_truth"], 
                                                ret["var_sel_reco"], 
                                                weights=ret["wgt_sel_reco"], 
                                                bins=[var_config.bins, var_config.bins]) # Fix: wrap them!
    eff = ret["nevts_sel_truth"] / ret["nevts_allmc"]
    response = get_response_matrix(reco_vs_true, eff)
    
    covariance_frac = genie_syst[var_config.var_save_name] + mcstat_syst[var_config.var_save_name]
    covariance = cov_from_fraccov(covariance_frac, ret["nevts_sel_reco"])
    
    bump_pos = 0.5
    bump_strength = 0.05
    bump_width = 0.1
    
    total_range = var_config.bins[-1] - var_config.bins[0]
    print(total_range)
    this_bump_pos = var_config.bins[0] + 0.5 * total_range
    print("Bump pos")
    print(this_bump_pos)
    
    weights_fake_data, weight_fakedata_signal_truth = fake_weight_obj.get_weights(test_name, bump_pos=this_bump_pos, bump_strength = bump_strength, bump_width = bump_width, var_config = var_config)
    fakedata_evt_df = mc_evt_df.copy()
    fakedata_evt_df[weight_col] *= weights_fake_data
    
    ret = signal_hists(mc_evt_df, mc_nu_df, var_config, return_data=True, plot=False)
    nevts_fakedata_reco, _ = np.histogram(ret["var_allsel_reco"], bins=var_config.bins, weights=weights_fake_data * ret["wgt_allsel_reco"])
    nevts_nomdata_signal_truth, _ = np.histogram(ret["var_allmc"], bins=var_config.bins, weights= ret["wgt_allmc"])
    nevts_fakedata_signal_truth, _ = np.histogram(ret["var_allmc"], bins=var_config.bins, weights=weight_fakedata_signal_truth*ret["wgt_allmc"])
    
    bins = var_config.bins
    bin_widths = np.diff(bins)
    fig, ax = plt.subplots()
    plt.hist(var_config.bin_centers, var_config.bins, weights=nevts_fakedata_reco, histtype="step", label="all selected events")
    plt.hist(var_config.bin_centers, var_config.bins, weights=nevts_nomdata_signal_truth, histtype="step", label="nominal MC, all signal")
    plt.hist(var_config.bin_centers, var_config.bins, weights=nevts_fakedata_signal_truth, histtype="step", label="alternative MC, all signal")
    plt.legend()
    plt.show();
    
    
    syst = np.sqrt(np.diag(covariance_frac))
    
    ret = overlay_hists(breakdown_type=breakdown_type,
                            mc_df=mc_evt_df,
                            data_df=fakedata_evt_df,
                            intime_df=None,
                            var_config=var_config,
                            plot_labels=plot_labels_hist,
                            ax_ylim_ratio=ax_ylim_ratio,
                            ratio=ratio,
                            syst=syst,
                            textloc=textloc,
                            approval=approval,
                            save_fig=False, 
                            save_name=path.join(save_fig_dir, "/{}_{}.png".format(var_config.var_save_name, breakdown_type)))
    
    
    measured = (ret["total_data"] - ret["total_mc_bkgd"]) 
    model = nevts_nomdata_signal_truth
    unfold = WienerSVD(response, model, measured, covariance, C_type, Norm_type)

    models = {"SBND Baseline Model": model, 
              "Fake Data": nevts_fakedata_signal_truth}
    
    save_name = "{}/{}_unfolded_event_rates".format(save_fig_dir, var_config.var_save_name)
    plot_unfolded_result(unfold, 
                         measured, 
                         models, 
                         var_config,
                         save_fig=save_fig, 
                         save_name=save_name,
                         closure_test=False)
    
    save_fig_name = "{}/{}-{}-add_smear".format(save_fig_dir, var_config.var_save_name, "closure_test")
    plot_heatmap(unfold["AddSmear"], 
                 var_config.bins, 
                 plot_labels=[var_config.var_labels[2], var_config.var_labels[1], "$A_c$"],
                 save_fig=False, 
                 save_name=save_fig_name)

In [ ]:
import numpy as np

# --- 1. Definir los bordes de los bins ---
# 10 bordes generan exactamente 9 bins

# Calcular los centros y los anchos para el histograma y las barras de error

bump_pos = 0.5
bump_strength = 0.05
bump_width = 0.1


test_name = "bump_0p5_0p1_0p1"
for var_config in var_configs:

    save_fig_dir = path.join(save_fig_base_dir, f"unfolding_fake_data_tests/fake_data_tests/{test_name}")
    if save_fig:
        if not path.exists(save_fig_dir):
            makedirs(save_fig_dir)
            
    bins = var_config.bins
    if var_config.var_save_name != "all_evts" and var_config.var_save_name != "num_protons":
        bins = np.linspace(bins[0], bins[-1], 10)

    # Calculate total range for the proportional width calculation we discussed earlier
    total_range = var_config.bins[-1] - var_config.bins[0]
    this_bump_pos = var_config.bins[0] + 0.5 * total_range
    
    # Using the proportional width formula to keep visual size consistent
    this_bump_widht = bump_width 

    print(f"Variable: {var_config.var_save_name}, Bump Width: {this_bump_widht}")
    
    fake_weight_obj = FakeDataWeights(mc_evt_df, mc_nu_df, var_config)
    weights_fake_data, weight_fakedata_signal_truth = fake_weight_obj.get_weights(
        test_name, bump_pos=this_bump_pos, bump_strength=bump_strength, 
        bump_width=this_bump_widht, var_config=var_config
    )

    bin_centers = (bins[:-1] + bins[1:]) / 2
    bin_widths = np.diff(bins)
    ret = signal_hists(mc_evt_df, mc_nu_df, var_config, return_data=True, plot=False)

    # --- 2. Extraer Variable y Pesos ---
    var_reco = mc_nu_df.loc[mc_nu_df.truth.nu_categ == "CC1pi", var_config.var_evt_truth_col]
    weights_fake = weight_fakedata_signal_truth
    
    # --- 3. Histogramas ---
    n_fake_reco, _ = np.histogram(ret["var_allmc"], bins=bins, weights=weights_fake*ret["wgt_allmc"])
    nevts_fakedata_signal_truth, _ = np.histogram(ret["var_allmc"], bins=bins, weights=ret["wgt_allmc"])
    
    n_plot_fake = n_fake_reco / bin_widths
    n_plot_truth = nevts_fakedata_signal_truth / bin_widths
    
    # --- 4. Plotting ---
    fig_fake, ax_fake = plt.subplots(figsize=(10, 8)) # Increased figure size slightly
    
    # Línea de la distribución (step)
    ax_fake.step(bins, np.append(n_plot_fake, n_plot_fake[-1]), 
                 where='post', color='darkorange', linewidth=3, label='Fake Data') # Thicker line
    ax_fake.step(bins, np.append(n_plot_truth, n_plot_truth[-1]), 
                 where='post', color='darkblue', linewidth=3, label='Base') # Thicker line
    
    ax_fake.plot(bin_centers, n_plot_fake, 'o', color='darkorange', markersize=6)
    
    # --- Aesthetics and Large Legend ---
    ax_fake.set_xlabel(var_config.var_plot_name + " " + var_config.var_unit, fontsize=16)
    ax_fake.set_ylabel(var_config.xsec_label, fontsize=16)
    ax_fake.set_ylim(0, np.max(n_plot_fake) * 1.4) # More headroom for the legend
    ax_fake.grid(True, linestyle='--', alpha=0.5)
    
    # legend customization:
    ax_fake.legend(
        fontsize=18,         # Sets the text size
        frameon=True,        # Ensures background box is visible
        edgecolor='black',   # Adds a border to the legend
        loc='upper right',   # Positioning
        fancybox=False       # Squared corners for a formal look
    )
    
    ax_fake.tick_params(axis='both', labelsize=14)
    plt.tight_layout()
    plt.show()

    save_name = f"{save_fig_dir}/{var_config.var_save_name}_bump_example.pdf"
    fig_fake.savefig(save_name, format='pdf', bbox_inches='tight')
    

In [ ]:

def plot_heatmap_general(matrix, x_vals, y_vals, var_config, plot_labels=["", "", ""], 
                         approval="internal", save_fig=False, save_name=None):
    n_rows, n_cols = matrix.shape
    is_pvalue = "p-value" in plot_labels[2].lower()

    # 1. Define Colormap and Normalization
    if is_pvalue:
        # Custom colormap: Red at 0, Yellow at 0.05, Green at 1.0
        colors = [
            (0.0, "red"),    # Significant
            (0.05, "yellow"), # Threshold
            (1.0, "green")    # Consistent
        ]
        cmap = mpl.colors.LinearSegmentedColormap.from_list("pvalue_cmap", colors)
        norm = mpl.colors.Normalize(vmin=0, vmax=1)
    else:
        cmap = plt.get_cmap("viridis")
        v_min = np.nanmin(matrix) if np.any(~np.isnan(matrix)) else 0
        v_max = np.nanmax(matrix) if np.any(~np.isnan(matrix)) else 1
        norm = mpl.colors.Normalize(vmin=v_min, vmax=v_max)

    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Imshow extent maps the matrix indices to the plot area
    im = ax.imshow(matrix, extent=[0, n_cols, 0, n_rows], origin="lower", 
                   aspect='auto', cmap=cmap, norm=norm)

    # 2. Colorbar and Exponent Logic
    cbar = plt.colorbar(im, shrink=0.8)
    flat_matrix = matrix[~np.isnan(matrix)]
    
    exponent = 0
    if not is_pvalue and flat_matrix.size > 0 and np.any(flat_matrix != 0):
        max_val = np.nanmax(np.abs(flat_matrix))
        exponent = int(np.floor(np.log10(max_val)))
        cbar.set_label(f"{plot_labels[2]} [10$^{{{exponent}}}$]", fontsize=18)
        cbar.ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda x, _: f"{x/10**exponent:.2f}"))
    else:
        cbar.set_label(plot_labels[2], fontsize=18)

    for i in range(n_rows):      # rows (y)
            for j in range(n_cols):  # columns (x)
                value = matrix[i, j]
                if not np.isnan(value):
                    # If values are small (e.g. < 0.01), use scientific notation
                    # Otherwise, use standard decimals
                    label = f"{value:.1e}" if abs(value) < 0.01 and value != 0 else f"{value:.2f}"
                    txt_color = get_text_color(value, cmap, norm)
                    
                    plt.text(
                        j + 0.5, i + 0.5,
                        label,
                        ha="center", va="center",   
                        color=txt_color,
                        fontsize=9 # Slightly smaller to fit scientific notation
                    )
                
    # 4. Axis Formatting 
    # X-axis labels: uses physical values from x_vals
    x_labels = [f"{x_vals[k]:.2f}" for k in range(n_cols)]
    y_labels = [f"{y_vals[k]:.2f}" for k in range(n_rows)]
    
    ax.set_xticks(np.arange(n_cols) + 0.5)
    ax.set_xticklabels(x_labels, rotation=45, ha="right")
    
    ax.set_yticks(np.arange(n_rows) + 0.5)
    ax.set_yticklabels(y_labels)

    # Update X-axis label with dynamic variable name and units
    x_axis_title = f"{var_config.var_plot_name} [{var_config.var_unit}]"
    ax.set_xlabel(x_axis_title, fontsize=20)
    ax.set_ylabel(plot_labels[1], fontsize=20)
    
    try:
        add_approval_text(approval, 0.95, 1.05, "right")
    except NameError:
        ax.set_title(f"{approval.upper()}", loc='right', alpha=0.5)

    plt.tight_layout()
    if save_fig and save_name:
        plt.savefig(f"{save_name}.png", bbox_inches='tight', dpi=200)
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
import warnings
import matplotlib as mpl
from os import path
from tqdm.auto import tqdm

# --- 2. Scan Configuration ---
n_steps = 10 # Increased for a better visual matrix, set to 2 for a quick test
pos_fractions = np.linspace(0, 1, n_steps)
height_fractions = np.linspace(0, 0.25, n_steps)

# Define edges for the plotting function (one more than n_steps)
pos_edges = np.linspace(0, 1, n_steps + 1)
height_edges = np.linspace(0, 0.25, n_steps + 1)
bump_width = 0.1
plot_dist = False

for var_config in var_configs:
    chi2_matrix = np.zeros((n_steps, n_steps))
    pval_matrix = np.zeros((n_steps, n_steps))
    
    # Pre-calculations
    ret_nom = signal_hists(mc_evt_df, mc_nu_df, var_config, return_data=True, plot=False)
    reco_vs_true, _, _ = np.histogram2d(ret_nom["var_sel_truth"], ret_nom["var_sel_reco"], 
                                        weights=ret_nom["wgt_sel_reco"], bins=[var_config.bins, var_config.bins])
    eff = ret_nom["nevts_sel_truth"] / ret_nom["nevts_allmc"]
    response = get_response_matrix(reco_vs_true, eff)
    covariance_frac = genie_syst[var_config.var_save_name] + mcstat_syst[var_config.var_save_name]
    covariance = cov_from_fraccov(covariance_frac, ret_nom["nevts_sel_reco"])
    model = np.histogram(ret_nom["var_allmc"], bins=var_config.bins, weights=ret_nom["wgt_allmc"])[0]

    fake_weight_obj = FakeDataWeights(mc_evt_df, mc_nu_df, var_config)

    # --- 3. The Silent 2D Scan Loop ---
    with open(os.devnull, 'w') as fnull:
        old_stdout = sys.stdout
        sys.stdout = fnull  
        try:
            pbar = tqdm(total=n_steps**2, desc=f"Scanning {var_config.var_save_name}", file=old_stdout)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                for i, bump_pos_frac in enumerate(pos_fractions):
                    for j, bump_height_frac in enumerate(height_fractions):
                        print(bump_pos_frac)
                        this_bump_pos = var_config.bins[0] + bump_pos_frac * (var_config.bins[-1] - var_config.bins[0])
                        this_bump_strength = bump_height_frac
                        test_name = f"bump_{this_bump_pos:.2f}_{this_bump_strength:.2f}"
                        
                        weights_fake_data, weight_fakedata_signal_truth = fake_weight_obj.get_weights(test_name, bump_pos=this_bump_pos, bump_strength = this_bump_strength, bump_width = bump_width, var_config = var_config)

                        fakedata_evt_df = mc_evt_df.copy()
                        fakedata_evt_df[('slc','wgt','','','','')] *= weights_fake_data
                        
                        ret_loop = signal_hists(mc_evt_df, mc_nu_df, var_config, return_data=True, plot=False)

                        if plot_dist: 
                            bins = var_config.bins
                            
                            # Calcular los centros y los anchos para el histograma y las barras de error
                            bin_centers = (bins[:-1] + bins[1:]) / 2
                            bin_widths = np.diff(bins)
                            # --- 2. Extraer Variable y Pesos ---
                            var_reco = mc_nu_df.loc[mc_nu_df.truth.nu_categ == "CC1pi",('truth', 'true_var', 'true_p_mu', '')]
                            weights_fake = weight_fakedata_signal_truth
                            
                            # --- 3. Histogramas ---
                            n_fake_reco, _ = np.histogram(ret_loop["var_allmc"], bins=bins, weights=weights_fake*ret_loop["wgt_allmc"])
                            nevts_fakedata_signal_truth, _ = np.histogram(ret_loop["var_allmc"], bins=bins, weights=ret_loop["wgt_allmc"])
                            
                            # --- 3.5 Calculate and Print Integrals ---
                            # Using the binned data (sum of counts per bin)
                            integral_fake = np.sum(n_fake_reco)
                            integral_truth = np.sum(nevts_fakedata_signal_truth)
                            
                            # Calculate the actual difference and the percentage
                            diff = integral_fake - integral_truth
                            percent_diff = (diff / integral_truth) * 100 if integral_truth != 0 else 0
                            
                            print("-" * 30)
                            print(f"INTEGRAL COMPARISON (Sum of Bins)")
                            print(f"Base Total: {integral_truth:.2f}")
                            print(f"Fake Total: {integral_fake:.2f}")
                            print(f"Difference: {diff:.2f}")
                            print(f"Percent increase: {percent_diff:.2f}%")
                            print("-" * 30)
                            
                            # Verification against your 'bump_strength' parameter
                            print(f"Target bump_strength: {bump_strength}")
                            
                            #n_plot_fake = n_fake_reco / bin_widths
                            #n_plot_truth = nevts_fakedata_signal_truth / bin_widths
                            
                            n_plot_fake = n_fake_reco 
                            n_plot_truth = nevts_fakedata_signal_truth
                            
                            # --- 4. Plotting ---
                            fig_fake, ax_fake = plt.subplots(figsize=(9, 7))
                            
                            # Línea de la distribución (step)
                            ax_fake.step(bins, np.append(n_plot_fake, n_plot_fake[-1]), 
                                         where='post', color='darkorange', linewidth=2, label='Fake Data (9 bins)')
                            ax_fake.step(bins, np.append(n_plot_truth, n_plot_truth[-1]), 
                                         where='post', color='darkorange', linewidth=2, label='Base (9 bins)')
                            
                            # Opcional: Agregar marcadores en los centros de los bins para mayor claridad
                            ax_fake.plot(bin_centers, n_plot_fake, 'o', color='darkorange', markersize=4)
                            
                            # Estética
                            ax_fake.set_xlabel(var_config.var_plot_name + " " + var_config.var_unit , fontsize=12)
                            ax_fake.set_ylabel(var_config.xsec_label, fontsize=12)
                            ax_fake.set_ylim(0, np.max(n_plot_fake) * 1.3)
                            ax_fake.grid(True, linestyle='--', alpha=0.5)
                            ax_fake.legend()
                            
                            plt.show()
                            
                        ov_ret = overlay_hists(mc_df=mc_evt_df, data_df=fakedata_evt_df, var_config=var_config, plot=False)
                        measured = ov_ret["total_data"] - ov_ret["total_mc_bkgd"]
                        nevts_fakedata_signal_truth, _ = np.histogram(ret_loop["var_allmc"], bins=var_config.bins, 
                                                                      weights=weight_fakedata_signal_truth * ret_loop["wgt_allmc"])

                        unfold = WienerSVD(response, model, measured, covariance, C_type, Norm_type)
                        fake_data_model_smeared = unfold['AddSmear'] @ nevts_fakedata_signal_truth
                        chi2_val, nodf, p_val = get_chi2(unfold['unfold'], fake_data_model_smeared, unfold['SystUnfoldCov'])
                        
                        chi2_matrix[j, i] = chi2_val/(len(fake_data_model_smeared))
                        pval_matrix[j, i] = p_val
                        pbar.update(1)
            pbar.close()
        finally:
            sys.stdout = old_stdout

    # --- 4. Plotting Results ---
    print(var_config.var_save_name)

    pos_physical = var_config.bins[0] + pos_fractions * (var_config.bins[-1] - var_config.bins[0])
    # --- Plot Chi2 ---
    plot_heatmap_general(
        chi2_matrix, 
        pos_physical,      # Now passing physical units for the X ticks
        height_fractions,  # Keeping height as fractions (unless you have a Y-unit too)
        var_config,        # New required argument for units and labels
        plot_labels=["", "Bump Height [Fraction]", r"$\chi^2/ndf$"], # X-label is now handled by var_config
        save_fig=True, 
        save_name=f"chi2_{var_config.var_save_name}"
    )
    
    # --- Plot P-Value ---
    plot_heatmap_general(
        pval_matrix, 
        pos_physical, 
        height_fractions, 
        var_config, 
        plot_labels=["", "Bump Height [Fraction]", "p-value"],
        save_fig=True, 
        save_name=f"pval_{var_config.var_save_name}"
    )